# EDA

In [2]:
import pandas as pd
import urllib.request
import io

columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','num']

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
files = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data'
}

headers = {'User-Agent': 'Mozilla/5.0 (compatible; DS-coursework/1.0)'}

def fetch_csv(url, colnames):
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp:
        raw = resp.read().decode('utf-8', errors='replace')
    return pd.read_csv(io.StringIO(raw), names=colnames, na_values='?')

# ---------- 1. FETCH & GABUNG ----------
dfs = []
for site, fname in files.items():
    try:
        df_site = fetch_csv(base_url + fname, columns)
        df_site['source'] = site
        dfs.append(df_site)
        print(f"{site}: {df_site.shape}")
    except Exception as e:
        print(f"{site}: GAGAL fetch -> {e}")

df_all = pd.concat(dfs, ignore_index=True)
df_all

cleveland: (303, 15)
hungarian: (294, 15)
switzerland: (123, 15)
va: (200, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,source
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,cleveland
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,cleveland
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,cleveland
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,cleveland
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,cleveland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,54.0,0.0,4.0,127.0,333.0,1.0,1.0,154.0,0.0,0.0,NaN,NaN,NaN,1,va
916,62.0,1.0,1.0,NaN,139.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,0,va
917,55.0,1.0,4.0,122.0,223.0,1.0,1.0,100.0,0.0,0.0,NaN,NaN,6.0,2,va
918,58.0,1.0,4.0,NaN,385.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,0,va


In [ ]:
# ---------- 2. CEK & DROP DUPLICATES ----------
# Duplikat dicek berdasarkan SELURUH kolom fitur (bukan 'source'),
# karena tidak ada ID pasien unik di dataset ini (lihat diskusi sebelumnya
# soal keterbatasan mendeteksi duplikat tanpa nomor RM).
feature_cols = [c for c in df_all.columns if c != 'source']

n_before = len(df_all)
dupe_mask = df_all.duplicated(subset=feature_cols, keep='first')
print(f"\nJumlah baris duplikat terdeteksi (exact match seluruh atribut): {dupe_mask.sum()}")

df_all = df_all[~dupe_mask].reset_index(drop=True)
print(f"Setelah drop duplicates: {df_all.shape} (dari {n_before})")

# CATATAN PENTING untuk laporan kalian:
# Karena tidak ada patient ID, ini adalah deteksi duplikat berbasis
# KECOCOKAN PERSIS seluruh nilai atribut -- berpotensi:
#   (a) melewatkan duplikat asli yang nilainya sedikit berbeda antar kunjungan
#   (b) TIDAK salah membuang dua pasien berbeda yang kebetulan identik semua
#       atributnya (risiko ini kecil karena ada >10 atribut kontinu/kategorikal
#       yang harus cocok semua, tapi tetap perlu didokumentasikan sebagai limitasi).

# ---------- 3. CEK MISSING VALUES ----------
print("\nMissing values per kolom:")
print(df_all.isna().sum())

missing_pct = df_all.isna().mean().sort_values(ascending=False)
print("\nPersentase missing per kolom:")
print((missing_pct * 100).round(1))

# ---------- 4. HANDLE MISSING VALUES ----------
# Strategi: drop kolom dengan missing value SANGAT tinggi (>50%),
# lalu drop baris yang masih punya missing value di kolom tersisa.
# (Alternatif: imputasi mean/median -- tapi untuk BASELINE REPLIKASI,
#  drop lebih transparan dan mudah dipertanggungjawabkan di laporan.)

high_missing_cols = missing_pct[missing_pct > 0.5].index.tolist()
print(f"\nKolom dengan missing >50% (akan di-drop): {high_missing_cols}")

df_clean = df_all.drop(columns=high_missing_cols)
n_before_dropna = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
print(f"Setelah dropna: {df_clean.shape} (dari {n_before_dropna})")

# ---------- 5. TARGET BINER ----------
# num asli berskala 0-4 (tingkat keparahan). Untuk replikasi binary
# classification seperti paper (Pass/Fail-style: normal vs diseased):
df_clean['target'] = (df_clean['num'] > 0).astype(int)
df_clean = df_clean.drop(columns=['num'])

print("\nDistribusi target akhir:")
print(df_clean['target'].value_counts())

print("\nDistribusi per source:")
print(df_clean.groupby(['source', 'target']).size())

# ---------- 6. SIMPAN ----------
out_path = "heart_disease_combined_clean.csv"
df_clean.to_csv(out_path, index=False)
print(f"\nDataset final tersimpan: {out_path} -> shape {df_clean.shape}")